
---
title: "Lab 1: PECARN TBI Data, Stat 215A, Fall 2026"
format:
  pdf:
    pdf-engine: xelatex
mainfont: "Times New Roman"
execute:
  echo: false
header-includes:
  - \usepackage{sectsty}
  - \allsectionsfont{\rmfamily}
  - \usepackage{titling}
  - \pretitle{\begin{center}\Huge\rmfamily\bfseries}
---

<div align="center">

### 1.  Introduction ###

</div>

Kupperman et al. [1] evaluated the dangers of CT imaging scans in children with head injuries. Standard practice has generally recommened CT scans to diagnose clinically-important traumatic brain injuries (ciTBIs). However, the radiation from the CT scans present their own danger of malignencies, especially for patients under 2 years old.

In this report, we look to make a model that can better identify whether a CT scan is needed for a traumatic brain injury. By identifying key factors that contribute to the severity of the injury, the model can create more specific inclusion criteria for patients who are most likley to need clinical treatment. 

This report first explains the dataset that was used in the analysis and the process to clean the data. Then, it outlines the steps of exploratory analysis and interesting findings. Lastly, it suggests a new model to update the clinical decision rule for traumatic brian injuries.

In [1]:
#|   output: false
# Import dependencies
import numpy as np
import pandas as pd
import matplotlib as plt
import importlib
import clean


In [2]:
#|   output: false
#  Load Data
pud = pd.read_csv(r"C:\Users\jcoll\Desktop\Everything\2026 Berkeley Fall\STAT 215A\stat-215a\lab1\data\TBI PUD 10-08-2013.csv")
pud_precleaned = pud.copy()



<div align="center">

## Part 1: EDA and Data Cleaning

</div>

<div align="center">

### 1.2  Data ###

</div>

For this analysis, the Pediatric Emergency Care Applied Research Network (PECARN) data from an observational cohort study was used. This set specifically looked at children younger than 18 who experienced minor head trauma in the past 24 hours and who were evaluated at 25 different departments. Practicioners used the Glasgow Coma Scale (GCS) score, a score from 1-15 to measure any decrease in consciousness. Lower scores indicate less consciousness and likley a more severe injury. This study focused on patients who presented with scores of 14-15, indicating a minor injury where a CT is more likley unnecessary, but the dataset contained patients with all scores. The dataset had 125 columns of data with descriptors of who filled out the form, patient characteristcs, patient symptoms, and patient outcomes. 


<div align="center">

### 1.3 Data Collection

</div>

Data for this study was collected by trained site investagators and emergency department physicians who recorded answers to the standardized questions. Amnesia, headache, and dizziness were not recorded for children younger than 2 years old. Data was verified with double and random triple data entry as well as annual site monitoring visits. The data was collected over the course of 2 years (2004-2006) by a variety of different types of practicioners across 25 different emergency departments. The study does not mention which location these emergency departments are in.

<div align="center">

### 1.4 Data Cleaning

</div>

The first step of data cleaning was just looking at all of the columns and ensuring understanding of their meaning. Part of this exercise resulted in a renaming of most columns in the dataset so they could be more descriptive. The new column names were for ease of explanatory use and understanding on the part of the user as to the implications of each column. Each column was then manually checked for unexpected values. For example, if we expected 1, 0, or 2, but there was a third option, that would have been flagged. In this case, the data aligned with their expected inputs. Then, prior to any modifications, there was an audit of missing, NaN, and empty variables in each column. A function scanned for true NaN values, blank strings, and placeholder strings like "-" or "NA". This function revieled that although the data was free of the commmon placeholder strings and empty elements, some columns had a high prevalence of true NaN values. The columns with the most NaN values are shown below. These variables are less likley to be helpful in our modeling stage.



In [3]:
importlib.reload(clean)
from clean import rename_columns, filter_low_severity

pud = rename_columns(pud)
pud_unfiltered_GCS = pud.copy() # will use later
pud = filter_low_severity(pud)

In [4]:
from clean import check_missing
check_missing(pud, placeholder_values=("NA", "N/A", "na", "n/a", ".", "-", "?")).head(5)


,NaN Value,Blank String,Placeholder String,Total Flagged
Ethnicity,15543,0,0,15543
Dizzy,15124,0,0,15124
Race,3088,0,0,3088
Acting_Normally,3010,0,0,3010
Observed_in_ED_Determined_CTDecision,2299,0,0,2299


The next step of the cleaning process ensured that there were no duplicate rows and dropping them if they existed. This dataset appeared quite clean and no rows needed to be dropped. Then, further exploration revealed that there were both float and integer types in the set. Considering the logic of each column, there was no need to keep any float values, so the entire dataframe was set to have integer values. NaNs were kept intact, however. 

In [5]:
#|   output: false
# Checking data type of dataframe columns
print("Pud before:")
print(pud.info())

for col in pud.columns:
    pud[col] = pd.to_numeric(pud[col], errors='coerce').astype('Int64')


Pud before:
<class 'pandas.DataFrame'>
Index: 42430 entries, 0 to 43398
Columns: 125 entries, PatNum to Clinically-Important_TBI
dtypes: float64(48), int64(77)
memory usage: 40.8 MB
None


In this dataset, we observed special codes that carried structural significance. The code "90" meant "Other," "91" meant pre-verbal/non-verbal patient, and "92" meant "Not Applicable." These codes were left alone since they carried meaning: the 90 option usually appeared where the options on the form did not capture reality (Race, destination after ED, physician certification, and injury mechanism all had 90 as an option), and it would not be correct to impute this, since it indicates alternate information. The 91 code appeared in the amnesia and headache columns, indicating the patient was non-verbal — meaning certain information about that patient is unavailable to us, important to know for modeling. The 92 code was often used to indicate a parent column had been marked 0, meaning details of the parent description were not applicable.

Parent-child relationships appeared frequently in the data. For example, the loss-of-consciousness parent column had a child asking about duration, and headache had children for severity and timing. If a patient did not have a headache (parent marked 0), the headache severity child should be marked 92. There is a fundamental difference between marking something not applicable versus leaving it blank or marking it 0: a blank indicates truly unknown information, while a 0 indicates a symptom surely is not present, even when a parent is indicated as existing. These subtle differences are why we tried to keep these instances distinct.

A function was made to audit parent/child relationships, using a map of these relationships checked in two ways. Under standard rules, if a parent column is marked "no" (0), all associated children should default to "not applicable" (92), while any active child response of "yes" (1) requires a corresponding "yes" (1) in the parent. When a parent is positive (1), its children typically record "yes" (1), "no" (0), or other valid positive codes (2, 3, 4, or 5); any NaN or 92 under a positive parent is treated as "no" (0), assuming omitted entries indicate the symptom's absence. A key exception applies to Loss_of_Conscious_History, which can take a value of 2 to indicate a suspected occurrence, allowing its children to house 92 or other specific values.


This check yeilded the following table:

In [6]:
importlib.reload(clean)

from clean import PARENT_CHILD_MAP
from clean import check_parent_child_logic

violations_dict = check_parent_child_logic(
    df=pud, 
    parent_child_map=PARENT_CHILD_MAP, 
    verbose=False
)

violations_summary = pd.DataFrame({
    "Parent Column": list(violations_dict.keys()),
    "Violations Found": [len(v) for v in violations_dict.values()]
})
violations_summary

,Parent Column,Violations Found
0,Loss_of_Consciousness_History,2285
1,Post-Traumatic_Seizure,89
2,Headache_During_Eval,1981
3,Vomit,1061
4,Altered_Mental_Status(AMS),0
5,Skull_Frax(SFx)_Palp,31
6,SFx_Basilar(Base),0
7,Hematoma,830
8,Trauma_Above_Clavicles,0
9,Neurological_Defficiency(ND),0


In [7]:
from clean import impute_parent_child_empirical

# 1. Loss of Consciousness (Has separate distributions for parent 1 and 2)
pud = impute_parent_child_empirical(
    pud,
    parent_col="Loss_of_Consciousness_History",
    child_cols="Duration_of_Consciousness_Loss",
    active_parent_vals=[1, 2],
)

# 2. Post-Traumatic Seizures (Two child columns)
pud = impute_parent_child_empirical(
    pud,
    parent_col="Post-Traumatic_Seizure",
    child_cols=["When_Seizure_Occured", "Length_of_Seizure"],
    active_parent_vals=[1],
)

# 3. Headaches (Two child columns)
pud = impute_parent_child_empirical(
    pud,
    parent_col="Headache_During_Eval",
    child_cols=["Headache_Severity", "Headache_Start_Time"],
    active_parent_vals=[1],
)

# 4. Vomit (Three child columns)
pud = impute_parent_child_empirical(
    pud,
    parent_col="Vomit",
    child_cols=["VomitNbr", "Vomit_Start_Time", "Last_Vomit_Ep_Time"],
    active_parent_vals=[1],
)

# Palpable Skull Fractures (One child column)
pud = impute_parent_child_empirical(
    pud,
    parent_col="Skull_Frax(SFx)_Palp",
    child_cols=["SFx_Palp_Depressed"],
    active_parent_vals=[1],
)

# 5. Hematoma (Two child columns)
pud = impute_parent_child_empirical(
    pud,
    parent_col="Hematoma",
    child_cols=["Hematoma_Location", "Hematoma_Size(SML)"],
    active_parent_vals=[1],
)

<div align="center">

##### 1.4.1 Meaning #####

</div>

The variables in this dataset generally indicate a symptom or factor that could contribute to a clinically important traumatic brain injury. Some columns are parents to others: the child columns describe or elaborate on the parent, and many are yes/no indicators of whether a patient has a given symptom. Other columns are basic descriptors of the patient, such as age, race, and ethnicity, while others describe who collected the data.

Looking at the parent/child logic and the differences between a 90, 91, 92, NaN, and 0 entry, we could already see that the meaning behind both variables and missing variables requires distinction.

Most variables are simple yes/no indicators, but several things could cause the recorded value to diverge from reality. Some uncertainty is built in when a patient is nonverbal — the amnesia and headache columns rely on questions that cannot be asked directly of a preverbal child. Cleaning also revealed that different practitioners filled out the form differently, and several variables are inherently subjective for whoever performs the intake: the GCS score requires judgment calls about whether sounds are incomprehensible or words inappropriate, and identifying "agitation" in altered mental status is similarly a matter of clinical opinion — one clinician may see fussing as normal, another as concerning enough to change the score. Only 4% of the data had a separate, independent assessment by another physician to check inter-rater reliability, so we assume there was no systemic bias in how clinicians filled out the form, since this underlies much of the subsequent analysis.

The data was also cleaned using secondary sources like medical records and morgue logs, assuming these records stayed consistent with the initial medical team's assessment.


<div align="center">

##### 1.4.2 Relevance #####

</div>

The dataset seeks to understand whether the clinical decision rule for clinically-important traumatic brain injuries is correctly filtering out children who do not need a CT scan. The reason this is the focus of the report is the large danger of the CT scan itself. The dataset we have is the same one used in this study to modify and refine the existing clinical rule on CT scans for children.

However, even though this dataset contains all of these variables, it is important to note their limitations, because that impacts how confidently the data can answer this question — especially since data quality is particularly important, rather than quantity. Something this study does not measure is malignancy. Specifically, it was not measured whether children who received a CT scan and did not have a brain injury, and had a higher rate of later developing a cancerous malignancy than those who did not have the CT scan. This is not reported in the study, and it would be important to collect if the goal is to lower the ultimate harm to patients. So, while this study can attempt to predict whether a CT scan was needed, it does not effectively link the CT scan to malignancy rates in these particular patients.

<div align="center">

##### 1.4.3 Comparability #####

</div>

It's interesting to note that the information collected in this study was taken from different hospitals and different emergency departments, and each individual hospital or emergency department could have differences in documentation, protocols, or even how clinicians score their patients. For example, in the GCS score, there could be different trends in how the GCS score is reported from hospital to hospital due to differing hospital cultures and practices. Hospitals may train their staff in different ways that could lead to systemic differences in how the data form is filled out. The data was collected over the span of 2 years, in which time protocols may have changed and office staff could have had turnover. This adds a bit less consistency to the process and is important to keep in mind for overall data trustworthiness. <br>

While cleaning the data, it was noticed that there was a difference in how many NaN values were in the rows of data by employee type. Below shows a table of these values and a graph of how NaN value occurance is slightly different between roles. If we knew which hospitals each employee came from, it would be interesting to see if there are any trends in missing values between hospitals.


In [8]:
from clean import empl_type_table, empl_type_mapping

empl_type_table(pud = pud, empl_type_mapping = empl_type_mapping)


,EmplType,Count
0,Faculty,21675
1,Fellow,3653
2,Nurse Practitioner,1948
3,Physician Assistant,1046
4,Resident,14090
5,NaN,18


In [9]:
importlib.reload(clean)
#| fig-width: 6
#| fig-height: 4.5
from clean import employee_missingness_plot, empl_type_mapping, certification_mapping
employee_missingness_plot(
    pud_precleaned=pud_precleaned,
    empl_type_mapping=empl_type_mapping,
    certification_mapping=certification_mapping
)

<Figure size 1800x1500 with 1 Axes>


Another thing worth noting for comparability is that there are parent and child columns: if a child column is checked, the parent column should inherently be checked as well. This is one of the reasons we ran the cleaning step to make sure this logic held. However, it will be important to note in the modeling that parent and child columns will be directly correlated, since they are not independent of each other.

<div align="center">

### 1.5 Findings

</div>

##### Finding 1 #####

Once the data was cleaned and the GCS was filtered for only less-severe cases, we found that 0.88% of trial participants had a ciTBI and 99.07% did not. This aligns well the the Kupperman study that indicated a 0.9% of patients had a ciTBI. The Kuppermann study team also used the existence of a palpable skull fracture as a key predictor variable for ciTBI in patients less than two years old, but used basilar skull fracture for patients two years old or older. This difference was interesting, so we decided to explore it further.

What we found was that palpable skull fracture is a substantially stronger predictor of ciTBI in children under two (23.0%) than in children two and older (11.1%). Conversely, basilar skull fracture is a substantially stronger predictor in children two and older (16.1%) than in children under two (7.0%). This pattern matches with the Kupperman study's choice of predictor by age group and each fracture type appears most useful in exactly the age group where the existing clinical rule uses it. However, it is important to note that there are very small fractions of patients who fall both on the lower end of the GCS scale and ultimately are diagnosed with a ciTBI. A larger sample may help.

In [10]:
#|   output: false
num_ciTBI = (len(pud[pud['Clinically-Important_TBI'] == 1]) / len(pud)) * 100
num_nonciTBI = (len(pud[pud['Clinically-Important_TBI'] == 0]) / len(pud)) * 100
print("The percent of trial participants with Clinically_Important Traumatic Brain Injuries in the study = ", num_ciTBI)
print("The percent of trial participants without Clinically_Important Traumatic Brain Injuries in the study = ", num_nonciTBI)

The percent of trial participants with Clinically_Important Traumatic Brain Injuries in the study =  0.886165448974782
The percent of trial participants without Clinically_Important Traumatic Brain Injuries in the study =  99.07141173697855


In [11]:
importlib.reload(clean)
#| fig-width: 6
#| fig-height: 4.5

from clean import plot_skull_fracture_citbi_by_age
rates = plot_skull_fracture_citbi_by_age(pud, sfx_col="Skull_Frax(SFx)_Palp",
                                       citbi_col="Clinically-Important_TBI",
                                       age_col="AgeTwoPlus",
                                       save_path="../figs/finding1_skull_fracture_citbi_by_age.png")


basilar_grouped = plot_skull_fracture_citbi_by_age(pud, sfx_col="SFx_Basilar(Base)",
                                   fracture_label="Basilar Skull Fracture",
                                   save_path="../figs/finding1b_basilar_skull_fracture_citbi_by_age.png")

<Figure size 1800x1200 with 1 Axes>

<Figure size 1800x1200 with 1 Axes>

##### Finding 2 #####
The next thing we did to look at potential columns for modeling was create a correlation matrix. We incorporated all of the columns from the Kuppermann study, as well as some other parent columns not necessarily directly correlated with a CT scan. Kuppermann's study used palpable skull fracture, altered mental status, loss of consciousness history, vomiting, headache during evaluation, and acting normally as the core predictors in the clinical rule. We decided to add post-traumatic seizure, trauma above the clavicles, neurological deficiency, and other substantial injury to see whether they either corroborate or conflict with the core predictors from the Kuppermann study.

We made the correlation matrix, and most of the values were fairly small, which makes sense because clinically important traumatic brain injuries did not make up a large portion of participants in the study, especially among patients with GCS scores of 14 or 15. It's interesting that palpable skull fracture only has a 0.05 correlation with clinically important TBI, despite the previous graph showing it can imply substantially increased ciTBI risk. This is likely because skull fracture itself is uncommon in the dataset, so it is not as strong of a predictor for the entire set of data. However, it does remain an important indicator for those who have it. 

What we do see is that the highest correlation in the correlation matrix is between altered mental status and acting normally. This is perhaps not surprising since the two are essentially measuring very similar things. In terms of correlation with clinically important TBI specifically, altered mental status also had the highest correlation, at 0.11.

In [12]:
importlib.reload(clean)

from clean import plot_predictor_correlation_matrix

corr = plot_predictor_correlation_matrix(pud)

<Figure size 2400x2100 with 2 Axes>

<div align="center">

### 1.6 Reality Check

</div>

As we went through the data cleaning process, we chose not to treat values such as 90, 91, or 92 as missing, since they carried true clinical meaning — a judgment call. If a value was entered as 92, for example, it indicates the column should not be considered at all, rather than being missing or not present, an important distinction. Our first assumption was that special codes are not meaningless and do not necessarily indicate a missing value: 90 indicated "Other," and 91 indicated a preverbal or nonverbal patient, both distinctions that carry meaning rather than a fundamental absence of something.

Our next assumption concerned parent and child columns. When a symptom's parent column indicated "No," we assumed all of its child columns should read "Not applicable" rather than simply being left as a non-value — this makes sense, since if a patient does not experience a parent symptom, the child symptoms would not be applicable to them.

When the parent column was active (1), we assumed child values should indicate either 0 or 1, since there should not be anything unknown if a parent is observed — if a child comes in with a headache, we assume the full protocol was followed and information on severity and start time would at least be approximated. Where a parent was active but child columns had NaN values, we used imputation to fill in the missing data, assuming the distribution of responses where data was fully filled would equal the distribution where the parent was indicated but children were NaN. This makes a big assumption that there was no systemic reason a clinician would mark a parent symptom and leave the rest as NaNs.

Our next modeling decision was splitting the data into two versions: one keeping all 92s intact for the tree models, and another recoding some of those 92s to 0 for the linear models. This assumes "not applicable" behaves like "no" for the purpose of the mathematical computation, but it is a simplification, since a distinction remains between information being not applicable and information being confirmed as non-existent in a clinical setting.

One thing that matches up consistently with the Kuppermann study is that restricting the data to just GCS scores of 14 or higher brought our ciTBI rate down from 1.8% to 0.886%, closely matching their reported rate of 0.9%. This validates the cleaning pipeline, and makes sense, since removing more severe cases with lower GCS scores should lower the ciTBI rate.

This generally aligns with the domain problem we are trying to solve, especially given how few instances of ciTBI exist in the data. Our cleaning process tried to minimize the deletion of rows and oversimplification of the data, remaining fairly conservative throughout. Due to the close match with the Kuppermann report, we believe our cleaned data passes the reality check. However, the imputation process assumed missingness is not systematic, which remains a limitation.

<div align="center">

### 1.7 Stability Check

</div>

Before the stability check, we decided to look at what would happen if we did not filter by GCS and did not remove the 969 entries where GCS was less than 14 — cases which were more severe. This was interesting because when we then looked at palpable skull fracture status and the percent of patients with a clinically important traumatic brain injury by age group, we found that not filtering by GCS changes what we would conclude for the clinical rule.

Without filtering by GCS score, palpable skull fracture status actually appears to be a slightly better indicator of ciTBI for patients two years and older (38.1%, n=147) than for patients younger than two (36.8%, n=76) — the two rates are close, with the older group marginally higher. This indicates that palpable skull fracture could have been a good choice for the tree model. But when we filter out those 969 values (just 2.3% of the dataset), we see a result congruent with the Kuppermann study: the rate for patients younger than two rises well above the rate for patients two and older (23.0%, n=61 vs. 11.1%, n=99), showing that palpable skull fracture is actually a substantially stronger indicator in the younger group once the more severe cases are removed. This makes sense, since we are filtering out the more severe cases, and those more severe cases are more likely to present with a symptom as significant as a skull fracture. Once we filter them out, it becomes clear that it is specifically patients younger than two years old who show a high incidence of palpable skull fracture leading to ciTBI, even in cases where their GCS score reflects a less severe status.

In [13]:
importlib.reload(clean)
#| fig-width: 6
#| fig-height: 4.5

from clean import clean_data, filter_low_severity, plot_skull_fracture_citbi_by_age, count_by_gcs_threshold


print("ORIGINAL (unfiltered population):")
grouped_original = plot_skull_fracture_citbi_by_age(pud_unfiltered_GCS, save_path="../figs/finding1_stability_original.pdf")

pud_gcs_filtered = filter_low_severity(pud_unfiltered_GCS)
print("\nPERTURBED (GCS >= 14 only):")
grouped_perturbed = plot_skull_fracture_citbi_by_age(pud_gcs_filtered, save_path="../figs/finding1_stability_gcs_filtered.pdf")

counts = count_by_gcs_threshold(pud_unfiltered_GCS)


ORIGINAL (unfiltered population):


<Figure size 1800x1200 with 1 Axes>


PERTURBED (GCS >= 14 only):


<Figure size 1800x1200 with 1 Axes>

GCS >= 14: 42,430
GCS < 14: 969


<div align="center">

## Part 2: Modeling

</div>

We used a logistic regression and a decision tree for this task. The logistic regression has a particular limitation. Because the linear model only takes in zeros and ones, it loses the distinction between 92s, which carry a real difference from zeros, as discussed earlier. The decision tree classifier, however, can treat 92 as its own separate category, allowing the model to account for that distinction of clinical meaning between a 92 and a 0.

In both models, we used six of Kuppermann's core clinical predictors: palpable skull fracture, altered mental status, loss of consciousness history, vomiting, headache during evaluation, and acting normally. Note that we did not split by age group as Kuppermann did nor did we include headache severity.

We started with the logistic regression for its interpretability and ease of use, which yielded unsurprising results. Altered mental status came out as the strongest predictor of clinically important TBI, followed by skull fracture, vomiting, and loss of consciousness history. At the other end, acting normally showed a negative coefficient with clinically important TBI. This mirrors some of Kupperman's findings. In a practical setting, if just these variables were considered, a doctor may have an easier time making a decision about a CT scan for potential ciTBI.

We then used a decision tree, which was able to preserve the real-world distinction between 92 and 0 embedded in the coding. The tree demonstrates that chances of ciTBI are lowest when none of the main six sympotms present, but risk dramatically increases when altered mental status, palpable skull fracture, and loss of consciousness are all present (25.8%). We chose a tree partly because Kuppermann also used a decision tree in their study, and because it allows us to capture nonlinear relationships within the model. We kept the max depth at three, which was a deliberate choice for interpretability — Kuppermann made a similar choice. This makes sense especially in a clinical setting, where a high-stress environment. A complicated clinical rule for assessing ciTBI may cause more trouble for practicioners using the rule.

In the tree, we can see, at its lowest 

Both models ultimately agreed that altered mental status ranked at the top, in both the logistic regression and the decision tree, and the coefficients matched clinical expectation. Again, we did not stratify by age, and we know from our stability check that age matters for at least the skull fracture predictor. Future modeling would need to account for this.

In [14]:
importlib.reload(clean)
from clean import generate_modeling_variants
df_tree, df_linear = generate_modeling_variants(pud)


In [15]:
importlib.reload(clean)
from clean import fit_logistic_regression, fit_decision_tree

logreg = fit_logistic_regression(df_linear, verbose=False)

coef_df = pd.DataFrame({
    "Predictor": ["Skull_Frax(SFx)_Palp", "Altered_Mental_Status(AMS)",
                  "Loss_of_Consciousness_History", "Vomit",
                  "Headache_During_Eval", "Acting_Normally"],
    "Coefficient": logreg.coef_[0]
}).sort_values("Coefficient", ascending=False)
coef_df

<Figure size 1500x900 with 1 Axes>

,Predictor,Coefficient
1,Altered_Mental_Status(AMS),1.505822
0,Skull_Frax(SFx)_Palp,1.080474
3,Vomit,0.800501
2,Loss_of_Consciousness_History,0.677250
4,Headache_During_Eval,0.002364
5,Acting_Normally,-0.617333


In [16]:
importlib.reload(clean)
from clean import fit_logistic_regression, fit_decision_tree
tree = fit_decision_tree(df_tree)

<Figure size 3000x2400 with 2 Axes>

<div align="center">

### 2.1 Discussion

</div>

Even though there were over 43,000 datapoints, data size was still an issue. Because ciTBI is so rare, there is an imbalance in our modeling: only about 0.9% of patients actually have a ciTBI, so our models are trained on data where "no ciTBI" is much more common, and will predict "no" much more frequently, even though the "yes" cases are the ones we must examine. Further, in our stability check, when we split the data into two age groups, the characteristics of individuals within the smaller buckets have a larger impact on the outcome. A larger dataset could mean larger subgroups with a more stable and trustworthy estimate of ciTBI rate.

The raw PECARN data and its collection process reflect the data/reality realm — this is where we get the subjectivity of clinical decisions, the different ways clinicians record data, and the variability across emergency departments.

Our cleaning pipeline, imputation, and regressions are part of the algorithms/models realm, where we try to make sense of the data we are given while maintaining consideration of the prior data/reality realm. In many cases there are unknowns and gaps between the two, which is why it is important to note the many assumptions happening in this realm.

How the model performs in the real world is part of the future data/reality realm. Our model cannot be overly complicated, or practitioners would not be able to use it effectively; however, it cannot be too simple, or it will send more patients than needed to a CT scan — it must have proper calibration. This is where a test set may come into play in a future study, to be compared against the current PECARN data results.

There is rarely a one-to-one correspondence between data and reality, though it can get close. The 90/91/92 codes had clinical meaning that made modeling difficult, requiring us to recode them for the linear regression model. There were often NaN values that could have resulted from negligence, unknowing, or missteps. There was even interpretability in how some columns were coded — for instance, the GCS score could be input differently by different practitioners, obscuring the real-life attributes each value represents.

Our data visualization did not present reality perfectly. One clear example was in the stability check: the graphs showed large bars where palpable and basilar skull fractures were the best indicators of ciTBI risk for different age groups, but those large bars had incredibly small n values relative to the dataset — thousands of non-ciTBI patients were grouped into the sliver of blue bars beside them.

Ultimately, this lab highlighted that a clinical decision rule may only be as trustworthy as the assumptions baked into it. We may never exactly replicate reality, but by listing judgment calls, recording logic leaps, and making a genuine effort to connect with the source of the data, we get closer to it.

<div align="center">

### 3.  Academic Pledge and Honesty Reflection ###

</div>

Professor Bin, <br><br>
&nbsp;&nbsp;&nbsp;&nbsp;For these assignments, I will maintain academic integrity and cite all sources that are used including collaborations and LLM use. In this assignment, Claude's Sonnet 5 was used to help me create a roadmap to complete the project, and Gemini's 3.5 Flash-Lite was used to help organize code into functions, debug, markdown formatting, and refining existing logic. For functions in clean.py assistance wwas used particularly for making visualizations. All suggestions from the LLM were reviewed and verified for alignment with intended logic.   <br>

&nbsp;&nbsp;&nbsp;&nbsp;Academic research honesty is necessary for the ultimate trustworthiness of future scientific knowledge. Being open about assumptions made and how conclusions were drawn can ensure that generalizations are minimized and the statistics can stay as objective as possible. In a world where it can be easy to skew the statistics to support a particular point of view, research performed with integrity can help minimize the misinterpretation of the data. Further, if science was made to understand life's complexities, then it is imperative that assumptions be observed, listed and challenged. Then, we can have a more robust foundation to build future research that can withstand the test of time. We can celebrate inquiry, notice where we fail, and not let that deter us from the pursuit of something a bit greater. Being honest about research means renouncing a fear of failure, being humble about findings, honoring the scientific method, and working toward the betterment of oneself as a scientist and the broader scientific community.

